# Live flesh runtime (R0 + R2)
ARDY text-to-motion -> the elf's leg -> muscle effort from the motion alone (ground reaction estimated from the motion,
MyoSuite static optimisation, the model's own activation dynamics) -> the small runtime (bulge + jiggle that stiffens with
effort, settings in docs/muscle-engine/r0_params.json) -> A (plain skinning) vs B side by side.  Plan:
docs/muscle-engine/runtime-plan.md.  Each step only runs when its outputs are missing (FORCE = True redoes it).

In [ ]:
import os, sys, glob, subprocess, html, base64
from IPython.display import display, HTML
REPO = r'E:\code\model-to-vrm'
PY = r'E:\splat\.venv\Scripts\python.exe'; PYB = r'E:\splat\.venv-body\Scripts\python.exe'; PYS = r'E:\splat\.venv-sim\Scripts\python.exe'
BLENDER = r'C:\Program Files\Blender Foundation\Blender 5.3\blender.exe'
def sh(cmd, log=None):
    """run a command, print its last lines; stop the notebook on failure"""
    r = subprocess.run(cmd, capture_output=True, text=True, encoding='utf-8', errors='replace', cwd=REPO)
    out = (r.stdout or '') + (r.stderr or '')
    if log: open(log, 'a', encoding='utf-8').write(out)
    print('\n'.join(l for l in out.splitlines()[-15:] if 'it/s]' not in l))
    if r.returncode: raise RuntimeError('failed: %s' % ' '.join(cmd[:4]))
def show(*patterns, width=320):
    """show images side by side (glob patterns), newest render of each file"""
    files = [f for p in patterns for f in sorted(glob.glob(p))]
    if not files: print('no pictures for', patterns); return
    tags = []
    for f in files:
        ext = os.path.splitext(f)[1][1:].replace('jpg', 'jpeg'); b = base64.b64encode(open(f, 'rb').read()).decode()
        tags.append('<figure style="display:inline-block;margin:4px"><img src="data:image/%s;base64,%s" width="%d"><figcaption style="font-size:11px">%s</figcaption></figure>' % (ext, b, width, html.escape(os.path.basename(f))))
    display(HTML(''.join(tags)))

PROMPT = 'A person jogs forward, then jumps up and lands.'; NAME = 'jogjump'; DURATION = 6
ARDY_OUT = r'E:\figures\ardy'; FD = r'E:\figures\elf_pipe\anatomy_dense'; MD = r'E:\figures\elf_pipe'; R0 = r'E:\figures\elf_pipe\rig_stage\r0'
PYA = r'E:\splat\.venv-ardy\Scripts\python.exe'; PYM = r'E:\splat\.venv-myo\Scripts\python.exe'
RS = lambda *p: os.path.join(REPO, 'research', 'rig_stage', *p); PARAMS = os.path.join(REPO, 'docs', 'muscle-engine', 'r0_params.json')
os.makedirs(R0, exist_ok=True); os.environ['PYTHONPATH'] = os.path.join(REPO, 'model_to_vrm', 'common')

## 1. ARDY clip (prompt encoded once on the CPU, ~16 GB RAM; motion on the GPU)

In [ ]:
FORCE = False
clip = os.path.join(ARDY_OUT, 'outputs', NAME + '.npz')
if FORCE or not os.path.exists(clip):
    sh([PYA, os.path.join(REPO, 'model_to_vrm', 'motion', 'ardy_run.py'), 'encode', PROMPT])
    sh([PYA, os.path.join(REPO, 'model_to_vrm', 'motion', 'ardy_run.py'), 'gen', PROMPT, '--model', 'soma', '--duration', str(DURATION), '--seed', '0', '--output', clip[:-4]])
print('clip', clip, os.path.exists(clip))

## 2. The clip on the elf's left leg (calibrated joint angles; ranges from myoLegs)

In [ ]:
FORCE = False
trf = os.path.join(R0, NAME + '_leg_transforms.npz'); rng = os.path.join(R0, 'myolegs_ranges.osim')
if not os.path.exists(rng): raise SystemExit('write myolegs_ranges.osim first (see r0_clip_activations.py notes)')
if FORCE or not os.path.exists(trf): sh([PY, RS('ardy_to_elf_leg_calibrated.py'), FD, R0, rng, clip])

## 3. R2: muscle effort from the motion alone, checked against published running EMG (red bars)

In [ ]:
FORCE = False
act = os.path.join(R0, 'act2_' + NAME + '.npz')
if FORCE or not os.path.exists(act): sh([PYM, RS('r0_clip_activations.py'), clip, ARDY_OUT, act])
sh([PY, RS('r2_activation_picture.py'), act, os.path.join(R0, 'r2_' + NAME + '.png'), os.path.join(REPO, 'docs', 'muscle-engine', 'r2_emg_windows.json')])
show(os.path.join(R0, 'r2_' + NAME + '.png'), width=900)

## 4. R0: the runtime (A plain skinning, B bulge + jiggle), its cost on the GPU

In [ ]:
FORCE = False
run = os.path.join(R0, NAME + '_run')
if FORCE or not os.path.exists(os.path.join(run, 'r0_skin.npz')): sh([PYS, '-u', RS('r0_flesh.py'), FD, MD, clip, trf, act, PARAMS, run])
print(open(os.path.join(run, 'r0_flesh.json')).read()[:1500]); sh([PYS, RS('r0_runtime_bench.py')])

## 5. Video: thigh and buttock close-up, A | B | B - A (mm); ~10 min on the GTX 1650

In [ ]:
FORCE = False
frames = os.path.join(R0, NAME + '_frames'); mp4 = os.path.join(R0, NAME + '_closeup.mp4')
if FORCE or not os.path.exists(mp4):
    os.environ['FOCUS'] = 'upper'; sh([PYS, RS('r0_render.py'), os.path.join(run, 'r0_skin.npz'), frames, 'backside', '1'])
    sh([PY, RS('r0_video.py'), frames, mp4, '60', '1'])
show(os.path.join(frames, 'f0168.png'), width=900); print('video:', mp4)